# News Article Classification using Word2Vec and Machine Learning

## Background

Text data is inherently unstructured and cannot be directly processed by machine learning models. To analyze and classify text effectively, it must first be transformed into a meaningful numerical representation.

Word Embeddings provide such a representation by capturing the contextual meaning of words, enabling models to understand relationships and patterns within the text. These representations can then be used to build classification models that automatically categorize text into predefined classes.



## Objective

To build a news classification model that predicts the category of a news article (World, Sports, Business, Sci/Tech) using Word2Vec embeddings for feature representation and applying Logistic Regression and Naive Bayes for classification.

## Dataset Description

For this task, we use the AG News dataset, a widely used benchmark dataset for text classification.

It contains news articles categorized into 4 classes:
1. World
2. Sports
3. Business
4. Sci/Tech

The dataset consists of:
- 120,000 training samples
- 7,600 test samples

Each data point includes:
- Text → News article/headline

- Label → Category of the news

| Label | Category |
| ----- | -------- |
| **0** | World    |
| **1** | Sports   |
| **2** | Business |
| **3** | Sci/Tech |


### Import Libraries

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
nltk.download('punkt')

import warnings
warnings.filterwarnings('ignore')

### Load Dataset

In [ ]:

dataset = load_dataset("ag_news")

train_data = dataset['train']
test_data = dataset['test']

In [3]:
labels = ["World", "Sports", "Business", "Sci/Tech"]

for i in range(5):
    text = dataset['train'][i]['text']
    label = dataset['train'][i]['label']
    
    print(f"\nText: {text}")
    print(f"Category: {labels[label]}")


Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Category: Business

Text: Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
Category: Business

Text: Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.
Category: Business

Text: Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the main pipeline in southern Iraq after\intelligence showed a rebel militia could strike\infrastructure, an oil official said on Saturday.
Category: Bus

### Data Preprocessing

In [4]:
def preprocess(text):
    text = text.lower()                      # Convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text)     # Remove special characters
    tokens = word_tokenize(text)             # Tokenization
    return tokens

#A list of tokenized sentences (list of lists of words)
train_tokens = [preprocess(text) for text in train_data['text']]
test_tokens = [preprocess(text) for text in test_data['text']]

y_train = train_data['label']
y_test = test_data['label']

### Train WORD2VEC Model (on train data only)


In [5]:
# Initialize Word2Vec model
w2v_model = Word2Vec(
    sentences=train_tokens,  # tokenized sentences
    vector_size=100,         # embedding size
    window=5,                # context window
    min_count=2,             # ignore rare words
    workers=4                # CPU cores
)

### Get Sentence into Vector Form

Word2Vec gives word-level embeddings, but our problem is:

**Sentence classification**

So we:

- Convert each news → list of word vectors
- Take average of vectors → sentence embedding

In [6]:
def get_sentence_vector(tokens, model):
    """
    This function converts a list of tokens (words) into a single sentence vector
    using a trained Word2Vec model.
    """
    
    vectors = []  # This will store word vectors for each token
    
    # Loop through each word in the tokenized sentence
    for word in tokens:
        
        # Check if the word exists in the Word2Vec vocabulary
        if word in model.wv:
            
            # If yes, get its vector representation and store it
            vectors.append(model.wv[word])
    
    # If no words from the sentence were found in the vocabulary
    if len(vectors) == 0:
        
        # Return a zero vector of same size as Word2Vec vectors
        return np.zeros(model.vector_size)
    
    # Otherwise, take the average of all word vectors
    # This gives a single fixed-size vector representing the sentence
    return np.mean(vectors, axis=0)


# Convert all training sentences into vectors
# Each sentence → one vector
X_train = np.array([
    get_sentence_vector(tokens, w2v_model)  # apply function on each sentence
    for tokens in train_tokens              # loop through all tokenized training sentences
])


# Convert all test sentences into vectors
X_test = np.array([
    get_sentence_vector(tokens, w2v_model)  # same process for test data
    for tokens in test_tokens
])

In [7]:
X_train

array([[ 0.02435963, -0.03599163, -0.14567642, ...,  0.23797625,
         0.1320819 ,  0.12127234],
       [ 0.40328473, -0.3918267 ,  0.24631463, ..., -0.47595894,
         0.15236183,  0.17550695],
       [ 0.28239998, -0.46998933, -0.29246286, ...,  0.22945124,
        -0.3549693 ,  0.12854862],
       ...,
       [ 0.18197118, -0.3401207 ,  0.5877033 , ..., -0.36168143,
         0.4579326 , -0.15317497],
       [-0.06768066, -0.5149926 ,  0.7449937 , ..., -0.49182   ,
         0.34278142, -0.18872721],
       [-0.07264671, -0.07364658,  0.8538271 , ..., -0.24727173,
         0.10186058, -0.24617968]], shape=(120000, 100), dtype=float32)

X_train is a matrix of 120,000 sentences where each sentence is represented as a 100-dimensional vector obtained by averaging its Word2Vec word embeddings.

120,000 → number of sentences (data points) in the training set
100 → number of features (dimensions) in each sentence vector (embedding size)

 Because machine learning models require fixed-size inputs, we convert every sentence—regardless of its length—into a 100-dimensional vector.
 
 X_test is a matrix of 7,600 sentences where each sentence is represented as a 100-dimensional vector.

### Train Multinomial Logistic Regression

In [8]:
clf = LogisticRegression(multi_class='multinomial',max_iter=1000)
clf.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'multinomial'


### Get Predictions and Evaluate

In [9]:
# PREDICTIONS
y_pred = clf.predict(X_test)

# EVALUATION
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8751315789473684

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.88      0.88      1900
           1       0.94      0.95      0.95      1900
           2       0.84      0.82      0.83      1900
           3       0.84      0.85      0.84      1900

    accuracy                           0.88      7600
   macro avg       0.87      0.88      0.87      7600
weighted avg       0.87      0.88      0.87      7600



### Train Naive Bayes

In [10]:
from sklearn.naive_bayes import GaussianNB

# Initialize model
nb_model = GaussianNB()

# Train model
nb_model.fit(X_train, y_train)

,priors,None
,var_smoothing,1e-09


### Get Predictions and Evaluate

In [11]:
# PREDICTIONS
y_pred = nb_model.predict(X_test)


# EVALUATION
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8342105263157895

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.86      0.86      1900
           1       0.89      0.94      0.91      1900
           2       0.84      0.73      0.78      1900
           3       0.76      0.80      0.78      1900

    accuracy                           0.83      7600
   macro avg       0.83      0.83      0.83      7600
weighted avg       0.83      0.83      0.83      7600



### **Conclusion:**
 
Logistic Regression outperforms Naive Bayes with higher accuracy (~88% vs ~83%) and better overall F1-scores, making it more suitable for Word2Vec-based news classification.


Naive Bayes remains a historically important NLP algorithm and performs strongly on sparse text representations such as Bag-of-Words and TF-IDF. However, for dense embedding-based representations like Word2Vec, discriminative models such as Logistic Regression often achieve superior performance.